# Citations and abstention

A grounded answer should carry structured provenance until the presentation layer. This notebook creates a cited answer, validates that citations refer to known chunks, and tests an explicit abstention policy.

## Policy flow

```text
FLOW (read top to bottom)

+--------------------+
| Question           |
+--------------------+
          |
          v
+---------------------+
| Retrieve candidates |
+---------------------+

Supporting paths:
  [Score threshold?] --yes--> [Evidence + citations]
  [Score threshold?] --no--> [Abstain with reason]
```

In [ ]:
from pathlib import Path
from examples.beginner.first_local_rag import load_chunks
from examples.beginner.citations import answer_with_citations, citations_are_retrieved, render_markdown

chunks = load_chunks(Path('../../examples/data/beginner-docs'))
grounded = answer_with_citations('What is an abstention?', chunks)
print(render_markdown(grounded))
assert citations_are_retrieved(grounded, chunks)

In [ ]:
unsupported = answer_with_citations('What is the capital of France?', chunks)
print(render_markdown(unsupported))
assert unsupported.abstained
assert unsupported.reason == 'insufficient-evidence'

A threshold is a policy choice, not a proof of truth. Measure supported questions, unsupported questions, false abstentions, and unsupported answers across a representative evaluation set. Also enforce authorization before retrieval; a valid citation can still be private or unauthorized.

In [ ]:
for threshold in (0.2, 0.8, 1.01):
    result = answer_with_citations('What is an abstention?', chunks, min_score=threshold)
    print(threshold, 'abstained=', result.abstained, 'citations=', len(result.citations))

## Exercise

Add claim-level citations: each claim should list the chunk IDs that support it. Write a test that rejects a citation ID not present in the retrieved set.